# Video recommender

Here is the final notebook that aims to implement the final model to recommend video based on the KuaiRec dataset. The goal is to use 3 different models to recommend video and address user cold start and diversity issues. Precision@K, NDCG@K and Diversity@K are the final choosen metrics to evaluate the model on the `small_matrix` test set given by the KuaiRec dataset.

The final model consist of using first the KNN recommender I implemented first in the `collaborative-models/` section. It is used when the user has liked less than a fixed threshold (10 for instance!). Then if the user has already watched and like some videos, we use a comination of both content based and collaborative based algorithms. As said before, I choose the Logistic regression model over the neural network one because of the simplicity of use and train that offers the logistic regression. For the collaborative model, I use the collaborative filtering model implemented previously. 

How to rank the output of both models? First, we put at the top of the list videos that are recommended in both models. Then we randomly choose a boolean value that follows a Bernouilli distribution. If the value is equal to true the we concatenate both lists by popularity in the ascending order. We do the opposite otherwise. This enables the model to have some serendipity and potentially add some diversity in the predictions as well.

Let's implement it!

## Import librairies

In [1]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import dataset_loader
import metrics
import numpy as np

MANUAL_RANDOM_SEED=42
EMBEDDINGS_PATH='./embeddings'

## Load and preprocess the test set

The test set is the `small_matrix` of the interactions between users and items given by the KuaiRec dataset. We need to preprocess a tiny bit the test ratings before using it (for instance defining if a user likes a video by thresholding the watch ratio around 0.8)

In [ ]:
# Load the interactions
interactions_test = dataset_loader.load_matrix('small_interactions')

Loading small_interactions...


In [4]:
interactions_test.head()

,user_id,video_id,play_duration,video_duration,time,date,timestamp,watch_ratio
0,14,148,4381,6067,2020-07-05 05:27:48.378,20200705.0,1.593898e+09,0.722103
1,14,183,11635,6100,2020-07-05 05:28:00.057,20200705.0,1.593898e+09,1.907377
2,14,3649,22422,10867,2020-07-05 05:29:09.479,20200705.0,1.593898e+09,2.063311
3,14,5262,4479,7908,2020-07-05 05:30:43.285,20200705.0,1.593898e+09,0.566388
4,14,8234,4602,11000,2020-07-05 05:35:43.459,20200705.0,1.593899e+09,0.418364


In [ ]:
# Check the shape to see if it matches the dataset documentation
interactions_test.shape

(4494578, 8)

In [8]:
# Same here, check the integrity of the dataset by counting the number
# of users in the test set
interactions_test['user_id'].nunique()

1411

In [9]:
def preprocess_interactions(interactions: pd.DataFrame):
    """
    Preprocess the test interactions. It defines whether if each user likes a video or not
    and keep only the relevant columns. Finally, we only want to keep the last interactions
    of a given user for the same videos otherwise it will introduce ambiguities for the models!
    """
    interactions['like'] = (interactions['watch_ratio'] >= 0.8).astype(int)
    user_ratings = interactions[['user_id', 'video_id', 'like', 'watch_ratio']]
    user_ratings = user_ratings.drop_duplicates(['user_id', 'video_id'], keep='last')
    return user_ratings

In [10]:
# Apply the preprocessing function
ratings_test = preprocess_interactions(interactions_test)
ratings_test.head()

,user_id,video_id,like,watch_ratio
0,14,148,0,0.722103
1,14,183,1,1.907377
2,14,3649,1,2.063311
3,14,5262,0,0.566388
4,14,8234,0,0.418364


## Load the train embeddings

In [11]:
videos_content: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_content.pkl')
videos_tags: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/videos_one_hot_tags.pkl')
avg_ratings_train: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_avg_ratings.pkl')
ratings_train: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_ratings.pkl')
user_encrypted: pd.DataFrame = pd.read_pickle(f'{EMBEDDINGS_PATH}/users_encrypted.pkl')

In [12]:
# Some videos in the test set are not present in the train one.
# This is because I drop duplicates as well as the non public videos
ratings_test = ratings_test[ratings_test['video_id'].isin(ratings_train['video_id'])]

## Implement the content based recommender model

I am now heading to the implementation of the logistic regression model to recommend videos. I will not detail anything more as the code below is just a implementation of the model built in the `content-based-models/logistic_regression` notebook. I use a main function to train and then generate the recommender function that can be used to predict videos.

In [ ]:
def generate_content_based_recommender(videos_tags: pd.DataFrame, videos_content: pd.DataFrame, avg_ratings: pd.DataFrame, ratings_train: pd.DataFrame):
    """
    Generate the content based recommender model using logistic regression

    Arguments
    ---
    videos_tags: pd.DataFrame
        video one hot encoding embedding for the video tags

    videos_content: pd.DataFrame
        matrix that holds the durations of each video

    avg_ratings: pd.DataFrame
        matrix that containes user average ratings for each video tag
        it must holds the average ratings using only the training interaction
    
    ratings_train: pd.DataFrame
        train interaction that is the big_matrix from the KuaiRec dataset

    Returns
    ---
    Recommender function to recommend videos
    """
    scaler = StandardScaler()

    def get_parameters(avg_ratings: pd.DataFrame, ratings: pd.DataFrame, predict: bool = False):
        """
        Get the inputs and outputs vectors to train and predict videos using logistic regression

        Arguments
        ---
        avg_ratings: pd.DataFrame
            matrix (for training phase) or vector (for prediction phase) of the user average ratings
        
        ratings: pd.DataFrame
            total ratings for building the output labels (it is not required for the prediction phase)

        predict: bool
            indicates if we are in the prediction phase or not

        Returns
        --- 
        X (model input), X_full (model input and indexes) and Y (output labels for training)
        """
        X_video = videos_tags.merge(videos_content, on='video_id')
        if not predict:
            ratings = ratings.drop('watch_ratio', axis=1)
            X_user = ratings.merge(avg_ratings, on='user_id')
            X_full = X_user.merge(X_video, on='video_id')
        else:
            X_full = avg_ratings.assign(key=1).merge(X_video.reset_index().assign(key=1), on='key').drop('key', axis=1)

        if predict:
            Y = None
            X = X_full.drop(['video_id', 'user_id'], axis=1)
        else:
            Y = X_full[['like']] 
            X = X_full.drop(['like', 'user_id', 'video_id'], axis=1)

        return X, X_full, Y

    # Training the model
    X, _, Y = get_parameters(avg_ratings, ratings_train)
    X = scaler.fit_transform(X)
    model = LogisticRegression()
    model.fit(X, Y.to_numpy().ravel())

    def recommend_content_based(user_avg_ratings: pd.DataFrame, user_previous_ratings: pd.DataFrame = None, filter_ratings: pd.DataFrame = None, k: int = 5) -> list[int]:
        """
        Recommend video using logistic regression

        Arguments
        ---
        user_avg_ratings: pd.DataFrame
            vector of dimension (1, n) that contains the average ratings of the user for each video tag
        
        user_previous_ratings: pd.DataFrame | None
            video the user has already watched previously

        filter_ratings: pd.DataFrame | None
            recommend only videos that are present in this matrix. This enables to test
            the model by using the test interactions matrix here so that we have label to compare
            and apply metrics on

        k: int = 5
            number of videos to recommend

        Returns
        ---
        List of k videos
        """
        X, X_full, _ = get_parameters(user_avg_ratings, None, predict=True)
        X = scaler.transform(X)
        probas = model.predict_proba(X)[:, 1].ravel()
        top_videos = X_full.iloc[probas.argsort()[::-1]]

        if filter_ratings is not None:
            top_videos = top_videos[top_videos['video_id'].isin(filter_ratings['video_id'])]
        elif user_previous_ratings is not None:
            top_videos = top_videos[~top_videos['video_id'].isin(user_previous_ratings['video_id'])]
        
        return top_videos['video_id'].to_list()[:k]
    
    return recommend_content_based

In [ ]:
# Generate the content based recommender function by using the training data
recommend_content_based = generate_content_based_recommender(videos_tags, videos_content[['video_duration']], avg_ratings_train, ratings_train)

### Evaluation of the content based model

We have now the recommender function that enables use to make prediction for specific user. We can now test the model over the test dataset to find out how it perform over unseen data. As training data is now huge, we should get way better results then before!

In [ ]:
# Get k=20 recommendations from the recommender
k = 20

# Initialize the recommendations matrix
recommendations: pd.DataFrame = None
avg_ratings_for_test = avg_ratings_train.reset_index()

# Loop through user that have interactions in the test set
for user_id in ratings_test['user_id'].unique():
    user_ratings = ratings_test[ratings_test['user_id'] == user_id]
    user_avg_ratings = avg_ratings_for_test[avg_ratings_for_test['user_id'] == user_id]

    # Recommend videos that only have interactions with that user in the test set
    # this is the purpose of using the filter_ratings parameter
    # We will be then able to compute metrics on those recommendations as we will have
    # the real watch ratio between the recommended videos and the user
    top_videos = recommend_content_based(user_avg_ratings, filter_ratings=user_ratings[['video_id']], k=k)
    top_videos = user_ratings[user_ratings['video_id'].isin(top_videos)]
    if recommendations is None:
        recommendations = top_videos
    else:
        recommendations = pd.concat([recommendations, top_videos])

In [37]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print()

Precision@5 -> 0.9220411055988661
NDCG@5 -> 0.9679947669863687

Precision@10 -> 0.9194188518781008
NDCG@10 -> 0.9674522032391519

Precision@15 -> 0.9220883534136546
NDCG@15 -> 0.9679298370526962

Precision@20 -> 0.9228206945428773
NDCG@20 -> 0.9687333817931754



### Interpretations

As we can read, we got way better results on when training the logistic regression over the whole train set. It generalize very well as we get 92% of precision. It seems that bad results are ofter located in the bottom of the recommendation list as the NDCG top up near 0.96 which is very great! However, I made the choice of not measuring the diversity at this stage because it would make the notebook to big and unreadable.

## Implementation of collaborative recommendations

Now we can move on implementing the first collaborative model, the one using the collaborative filtering method as we saw in the `collaborative-models/collaborative_filtering` notebook. I used the same architecture for this model with a main function that trains over the train set and then returns a recommender function to make predictions.

In [ ]:
def generate_collaborative_recommender(collab_ratings: pd.DataFrame):
    """
    Generate the recommender function to recommend videos using collaborative filtering
    
    Arguments
    ---
    ratings: pd.DataFrame
        collab_ratings interactions between users and videos
    """

    video_labels, video_uniques = pd.factorize(collab_ratings['video_id'])
    user_labels, user_uniques = pd.factorize(collab_ratings['user_id'])
    
    collab_ratings['video_index'] = video_labels
    collab_ratings['user_index'] = user_labels
    videos_idx_mapping = dict(zip(range(len(video_uniques)), video_uniques))
    users_idx_mapping = dict(zip(user_uniques, range(len(user_uniques))))

    def build_matrices(ratings: pd.DataFrame):
        """
        Create the matrices Y (actual ratings) and R (if the user as interacted with the video)
        to learn the feature vectors X, W and b
        
        Arguments
        ---
        ratings: pd.DataFrame
            training interactions between users and videos
            I use the watch ratio as the value to learn

        Returns
        ---
        Matrices Y and R as well as the parameters norm and mean used for scaling the data
        """
        Y = ratings.pivot_table(values='watch_ratio', columns='user_index', index='video_index').fillna(-1)
        Y = Y.to_numpy()
        R = np.where(Y == -1, 0, 1)
        Y_mean = (np.sum(Y * R, axis=1) / (np.sum(R, axis=1) + 1e-12)).reshape(-1, 1)
        Y_norm = Y - Y_mean
        return Y, R, Y_norm, Y_mean

    def cost_func(X, W, b, Y, R, lambda_):
        """
        Custom cost function used by Tensorflow to learn the feature vectors
        """
        j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y) * R
        J = 0.5 * tf.reduce_sum(j**2) + (lambda_ / 2) * (
            tf.reduce_sum(X**2) + tf.reduce_sum(W**2)
        )
        return J

    # Building inputs
    Y, R, Y_norm, Y_mean = build_matrices(collab_ratings)
    num_videos, num_users = Y.shape
    num_features = 140

    # Training phase
    tf.random.set_seed(MANUAL_RANDOM_SEED)
    W = tf.Variable(tf.random.normal((num_users, num_features), dtype=tf.float64), name="W")
    X = tf.Variable(
        tf.random.normal((num_videos, num_features), dtype=tf.float64), name="X"
    )
    b = tf.Variable(tf.random.normal((1, num_users), dtype=tf.float64), name="b")
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-1)

    for iter in range(300):
        with tf.GradientTape() as tape:
            cost_value = cost_func(X, W, b, Y_norm, R, lambda_=0.8)
        grads = tape.gradient(cost_value, [X, W, b])
        optimizer.apply_gradients(zip(grads, [X, W, b]))
        if iter % 20 == 0:
            print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

    # Build the predictions matrix that approximate the actual matrix of ratings
    predictions = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy() + Y_mean

    def recommend_collaborative(user_id: int, previous_user_ratings: pd.DataFrame = None, filter_ratings: pd.DataFrame = None, k: int = 5) -> list[int]:
        """
        Recommend videos using collaborative filtering
        
        Arguments
        ---
        user_id: int
            id of the user to recommend video
            
        user_previous_ratings: pd.DataFrame | None
            video the user has already watched previously

        filter_ratings: pd.DataFrame | None
            recommend only videos that are present in this matrix. This enables to test
            the model by using the test interactions matrix here so that we have label to compare
            and apply metrics on

        k: int = 5
            number of videos to recommend

        Returns
        ---
        List of k recommended videos 
        """
        user_index = users_idx_mapping[user_id]
        user_predictions = predictions[:, user_index]
        user_predictions = pd.DataFrame(np.column_stack((collab_ratings['video_index'].unique(), user_predictions)), columns=['video_index', 'prediction'])
        user_predictions = user_predictions.sort_values('prediction', ascending=False)
        user_predictions['video_id'] = user_predictions['video_index'].map(videos_idx_mapping)

        if filter_ratings is not None:
            user_predictions = user_predictions[user_predictions['video_id'].isin(filter_ratings['video_id'])]
        elif previous_user_ratings is not None:
            user_predictions = user_predictions[~user_predictions['video_id'].isin(previous_user_ratings['video_id'])]

        return user_predictions['video_id'].to_list()[:k]

    return recommend_collaborative

In [ ]:
# Generate the recommender function
recommend_collaborative = generate_collaborative_recommender(ratings_train)

Training loss at iteration 0: 721995146.6
Training loss at iteration 20: 18819035.1
Training loss at iteration 40: 11994994.7
Training loss at iteration 60: 9731594.5
Training loss at iteration 80: 8341700.8
Training loss at iteration 100: 7501190.3
Training loss at iteration 120: 6956649.9
Training loss at iteration 140: 6574182.3
Training loss at iteration 160: 6290588.8
Training loss at iteration 180: 6072023.7
Training loss at iteration 200: 5898614.6
Training loss at iteration 220: 5758184.5
Training loss at iteration 240: 5642760.5
Training loss at iteration 260: 5546779.0
Training loss at iteration 280: 5466168.2


### Evaluate the model

In [38]:
k = 20
recommendations: pd.DataFrame = None

ratings_test['video_index'] = np.zeros(len(ratings_test), dtype=int)
ratings_test['user_index'] = np.zeros(len(ratings_test), dtype=int)

for user_id, user_data in ratings_test.groupby('user_id'):
    top_videos = recommend_collaborative(user_id, filter_ratings=user_data, k=k)
    top_videos = user_data[user_data['video_id'].isin(top_videos)]
    if recommendations is None:
        recommendations = top_videos
    else:
        recommendations = pd.concat([recommendations, top_videos])

In [39]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print()

Precision@5 -> 0.8259390503189227
NDCG@5 -> 0.9263193500181701

Precision@10 -> 0.8162296243798725
NDCG@10 -> 0.9255493546320246

Precision@15 -> 0.8105362626978503
NDCG@15 -> 0.926005768464309

Precision@20 -> 0.8046775336640679
NDCG@20 -> 0.9268146050574408



### Interpretations

To recall, I got a Precision@5 equals to 69% when I first built the model and splitting the training set to form another validation set. This model is actually very sensible of the amount of information we have: the more information about the interactions between the users and videos we have, the more the approximation of the actual ratings matrix is precise. It confirms this fact as we get a Precision@20 above 80%!

## Implement KNN cold start collaborative model

Let's now implement the collaborative model using KNN for addressing the issue of cold start when a new user want to watch videos. The implementation is different than the last two models as it does not require a training phase: it is a straightforward model by just predicting the video to recommend.

In [ ]:
def filter_ratings_users_for_knn(users: pd.DataFrame, ratings: pd.DataFrame):
    """
    Filter the full interactions matrix (test + train sets) to optimize and speed up the
    recommendation of videos

    Arguments
    ---
    users: pd.DataFrame
        list of users content vectors

    ratings: pd.DataFrame
        interactions (or "ratings") matrix between users and videos
    """
    users = users[users['user_active_degree'] == 'high_active']
    ratings = ratings[ratings['user_id'].isin(users.index)]
    ratings = ratings[ratings['like'] == 1]
    users = users.drop('user_active_degree', axis=1)
    return ratings, users

def recommend_collaborative_cold_start(user_id: int, user_data: pd.DataFrame, users: pd.DataFrame, user_ratings: pd.DataFrame, filter_ratings: pd.DataFrame = None, k: int = 5):
    """
    Recommend videos using a KNN approach
    
    Arguments
    ---
    user_id: int
        the user that will get the recommendation list
    
    users: pd.DataFrame
        the users to compare our actual user with
    
    user_ratings: pd.DataFrame
        the video that the users has already seen (or rated)
    
     filter_ratings: pd.DataFrame | None
            recommend only videos that are present in this matrix. This enables to test
            the model by using the test interactions matrix here so that we have label to compare
            and apply metrics on

    k: int = 5
        number of videos to recommend

    Returns
    ---
    List of k recommended videos 
    """
    users = users[users.index != user_id]

    similarities = cosine_similarity([user_data], users).ravel()
    similarities = similarities.argsort()[::-1][:10]
    similar_users = users.iloc[similarities]

    similar_user_ratings = user_ratings[user_ratings['user_id'].isin(similar_users.index)]

    if filter_ratings is not None:
        similar_user_ratings = similar_user_ratings[similar_user_ratings['video_id'].isin(filter_ratings[filter_ratings['user_id'] == user_id]['video_id'])]

    return similar_user_ratings['video_id'].value_counts().index[:k].to_list()

In [ ]:
# Concat the test and train ratings as there is no "train/test" sets for this model
full_users_ratings = pd.concat([ratings_test, ratings_train])
full_users_ratings = full_users_ratings.drop_duplicates(['video_id', 'user_id'], keep='last')
full_users_ratings = full_users_ratings.set_index('user_id')

In [46]:
knn_ratings, knn_users = filter_ratings_users_for_knn(user_encrypted, full_users_ratings.reset_index())

### Evaluate the model

In [ ]:
recommendations = None

for user_id, user_data in user_encrypted.groupby('user_id'):
    if user_id % 20 == 0:
        print(f'Current user is: {user_id}')
    user_data = user_data.drop('user_active_degree', axis=1).to_numpy().ravel()
    recommendation = recommend_collaborative_cold_start(user_id, user_data, knn_users, knn_ratings, filter_ratings=ratings_test, k=20)
    video_recommended = full_users_ratings.loc[user_id]
    video_recommended = video_recommended[video_recommended['video_id'].isin(recommendation)]

    if recommendations is None:
        recommendations = video_recommended
    else:
        recommendations = pd.concat([recommendations, video_recommended])

Current user is: 0
Current user is: 20
Current user is: 40
Current user is: 60
Current user is: 80
Current user is: 100
Current user is: 120
Current user is: 140
Current user is: 160
Current user is: 180
Current user is: 200
Current user is: 220
Current user is: 240
Current user is: 260
Current user is: 280
Current user is: 300
Current user is: 320
Current user is: 340
Current user is: 360
Current user is: 380
Current user is: 400
Current user is: 420
Current user is: 440
Current user is: 460
Current user is: 480
Current user is: 520
Current user is: 540
Current user is: 560
Current user is: 580
Current user is: 600
Current user is: 620
Current user is: 640
Current user is: 660
Current user is: 680
Current user is: 700
Current user is: 720
Current user is: 740
Current user is: 760
Current user is: 780
Current user is: 800
Current user is: 820
Current user is: 840
Current user is: 860
Current user is: 880
Current user is: 900
Current user is: 920
Current user is: 940
Current user is: 96

In [ ]:
recommendations = recommendations.reset_index()

# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print()

Precision@5 -> 0.8053175775480059
NDCG@5 -> 0.9150236932101558

Precision@10 -> 0.8028803545051698
NDCG@10 -> 0.9169800153909738

Precision@15 -> 0.8012801575578533
NDCG@15 -> 0.9185262680720262

Precision@20 -> 0.7998892171344166
NDCG@20 -> 0.9202131004482004



### Interpretations of the metrics

We get slightly better results in terms of precision (+10% here compare to using only the train set). It seems that the high density of interactions that the small matrix bring here helps to improve the knn performance a lot.

## Implement the final recommender algorithm

Here is the final implementation of the final recommender system algorith. It follow the rules that I explained in the introduction of this notebook!

In [ ]:
def recommend_videos(user_id: int, user_ratings: pd.DataFrame = None, filter_ratings: pd.DataFrame = None, cold_start_user_threshold: int = 10, k: int = 10):
    """ 
    Recommend videos
    
    Arguments
    ---
    user_id: int
        the user that will get the recommendations
        
    user_ratings: pd.DataFrame
        the video that the users has already seen (or rated)
    
    filter_ratings: pd.DataFrame | None
        recommend only videos that are present in this matrix. This enables to test
        the model by using the test interactions matrix here so that we have label to compare
        and apply metrics on

    cold_start_user_threshold: int = 10
        set the minimal number of video the user need to like before applying content based and collaborative filtering recommendations
    
    k: int = 10
        number of video to recommend

    Returns
    ---
    List of k recommended videos 
    """
    cold_start_user = False
    if user_ratings is None or len(user_ratings[user_ratings['like'] == 1]) < cold_start_user_threshold:
        cold_start_user = True
    
    if cold_start_user:
        if user_id not in user_encrypted.index:
            return []
        user_data = user_encrypted.loc[user_id]
        return recommend_collaborative_cold_start(user_id, user_data, knn_users, knn_ratings, filter_ratings=filter_ratings, k=k)
    else:
        user_avg_ratings = avg_ratings_for_test[avg_ratings_for_test['user_id'] == user_id]
        top_collab_videos = recommend_collaborative(user_id, filter_ratings=filter_ratings, k=k)
        top_collab_videos = videos_content.loc[top_collab_videos].reset_index()
        top_content_videos = recommend_content_based(user_avg_ratings, filter_ratings=filter_ratings, k=k)
        top_content_videos = videos_content.loc[top_content_videos].reset_index()

        # Put common videos in both recommendation outputs on top
        common_videos = pd.merge(top_content_videos, top_collab_videos, on='video_id', how='inner')
        common_videos = common_videos[['video_id']].drop_duplicates()
        all_videos = pd.concat([top_content_videos, top_collab_videos]).drop_duplicates('video_id')
        rest = all_videos[~all_videos['video_id'].isin(common_videos['video_id'])]
        rest_sorted = rest.sort_values(['show_cnt', 'play_cnt', 'share_cnt'], ascending=np.random.binomial(1, 0.5))
        top_videos = pd.concat([
            common_videos.merge(all_videos, on='video_id', how='left'),
            rest_sorted
        ])

        return top_videos['video_id'].tolist()[:k]

In [ ]:
def perform_test_for_user(user_id: int = 14):
    """
    Recommend video on the test set only
    """
    user_ratings = ratings_train[ratings_train['user_id'] == user_id]
    filter_ratings = ratings_test[ratings_test['user_id'] == user_id]
    recommended_videos = recommend_videos(user_id, user_ratings, filter_ratings=filter_ratings, k=20)
    recommended_videos_actual_rating = filter_ratings[filter_ratings['video_id'].isin(recommended_videos)]
    return recommended_videos_actual_rating

In [ ]:
# Recommendations for user 14 on the test set
perform_test_for_user()

,user_id,video_id,like,watch_ratio
85,14,8298,1,1.370336
171,14,8366,1,1.387273
193,14,8186,1,1.678448
379,14,3923,1,1.939164
397,14,9815,1,1.046427
515,14,7070,0,0.757458
830,14,314,1,3.774698
891,14,9582,1,1.427536
894,14,637,1,0.995672
1048,14,9907,1,1.073722


In [187]:
recommendations = None
users = ratings_test['user_id'].unique()

for i in range(len(users)):
    if i % 20 == 0:
        print(f'Progress {i} / {len(users)}')

    user_id = users[i]
    video_recommended = perform_test_for_user(user_id)

    if len(video_recommended) == 0:
        print(f'No recommendations for user {user_id}')
    
    if recommendations is None:
        recommendations = video_recommended
    else:
        recommendations = pd.concat([recommendations, video_recommended])

Progress 0 / 1411
Progress 20 / 1411
Progress 40 / 1411
Progress 60 / 1411
Progress 80 / 1411
Progress 100 / 1411
Progress 120 / 1411
Progress 140 / 1411
Progress 160 / 1411
Progress 180 / 1411
Progress 200 / 1411
Progress 220 / 1411
Progress 240 / 1411
Progress 260 / 1411
Progress 280 / 1411
Progress 300 / 1411
Progress 320 / 1411
Progress 340 / 1411
Progress 360 / 1411
Progress 380 / 1411
Progress 400 / 1411
Progress 420 / 1411
Progress 440 / 1411
Progress 460 / 1411
Progress 480 / 1411
Progress 500 / 1411
Progress 520 / 1411
Progress 540 / 1411
Progress 560 / 1411
Progress 580 / 1411
Progress 600 / 1411
Progress 620 / 1411
Progress 640 / 1411
Progress 660 / 1411
Progress 680 / 1411
Progress 700 / 1411
Progress 720 / 1411
Progress 740 / 1411
Progress 760 / 1411
Progress 780 / 1411
Progress 800 / 1411
Progress 820 / 1411
Progress 840 / 1411
Progress 860 / 1411
Progress 880 / 1411
Progress 900 / 1411
Progress 920 / 1411
Progress 940 / 1411
Progress 960 / 1411
Progress 980 / 1411
Progre

In [205]:
recommendations.head()

,index,user_id,video_id,like,watch_ratio
0,85,14,8298,1,1.370336
1,171,14,8366,1,1.387273
2,193,14,8186,1,1.678448
3,379,14,3923,1,1.939164
4,397,14,9815,1,1.046427


In [211]:
# Select columns in the recommendation dataframe to compute the diversity metric on
if 'index' in recommendations.columns:
    recommendations = recommendations.drop('index', axis=1)

recommendations = recommendations.merge(videos_tags, on='video_id')
recommendations = recommendations.merge(videos_content.iloc[:, 2:], on='video_id')

recommendations.head()

,user_id,video_id,like,watch_ratio,tag_0,tag_1,tag_2,tag_3,tag_4,tag_5,...,tag_28,tag_29,tag_30,show_cnt,play_cnt,valid_play_cnt,like_cnt,comment_cnt,share_cnt,video_duration
0,14,8298,1,1.370336,0,0,0,0,0,0,...,0,0,0,580837,453961,352644,7593,112,28,3270
1,14,8366,1,1.387273,0,0,0,0,0,0,...,0,0,0,531519,520237,414062,2856,35,4,3300
2,14,8186,1,1.678448,0,0,0,0,0,0,...,1,0,0,505597,520205,359667,11178,180,101,4923
3,14,3923,1,1.939164,0,0,0,0,0,0,...,0,0,0,13617621,13178925,7911486,1001936,19475,9484,5934
4,14,9815,1,1.046427,0,0,0,0,0,0,...,0,0,0,684997,701548,486873,3685,18,5,3834


In [212]:
# Select columns in the recommendation dataframe to compute the diversity metric on
video_content_columns = recommendations.columns[4:]
video_content_columns

Index(['tag_0', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7',
       'tag_8', 'tag_9', 'tag_10', 'tag_11', 'tag_12', 'tag_13', 'tag_14',
       'tag_15', 'tag_16', 'tag_17', 'tag_18', 'tag_19', 'tag_20', 'tag_21',
       'tag_22', 'tag_23', 'tag_24', 'tag_25', 'tag_26', 'tag_27', 'tag_28',
       'tag_29', 'tag_30', 'show_cnt', 'play_cnt', 'valid_play_cnt',
       'like_cnt', 'comment_cnt', 'share_cnt', 'video_duration'],
      dtype='object')

In [213]:
# Top videos to retrieve
kx = [5, 10, 15, 20]

for k in kx:
    # I use the metrics python script where I implemented all the relevant metrics
    # that I want to use. See metrics.py for more information
    precision = metrics.precision_at_k(recommendations, k)
    ndgc = metrics.ndcg_at_k(recommendations, k)
    diversity = metrics.inter_list_diversity_at_k(recommendations, video_content_columns, k)

    print(f'Precision@{k} -> {precision}')
    print(f'NDCG@{k} -> {ndgc}')
    print(f'ILD@{k} -> {diversity}')
    print()

Precision@5 -> 0.8498936924167257
NDCG@5 -> 0.9396318988913205
ILD@5 -> 0.9758283755898477

Precision@10 -> 0.8411055988660526
NDCG@10 -> 0.938063667202497
ILD@10 -> 0.9737102206798711

Precision@15 -> 0.8337349397590362
NDCG@15 -> 0.9379246987472473
ILD@15 -> 0.9777303084078894

Precision@20 -> 0.8282069454287738
NDCG@20 -> 0.9380300351558475
ILD@20 -> 0.9864852652432687



## Conclusion

The model has an average precision of 84%, is able to great video in order because NDCG is really high and it recommend extremly diverse videos (the diversity metric is almost equal to 1!). I feel like the combining the models in that way clearly proove that each model contribute on its own to giving great recommendations. I think using just a simple logistic regression would have been also very powerful but it does not address cold start issues with the user and the diversity of recommended videos would not have been really wide.

I hope you liked my proposal solution of recommending videos using KuaiRec dataset. I tried my best of thinking about which metrics, which features and which models were the best to use. I really work hard on giving some intuitions of what I was doing. The project really enabled me to have more confidence of using AI/ML librairies and be now extremly familiar on how recommend systems work today!